# code copied from the first part of Step 10

In [1]:
import config

In [2]:
#Import the necessary modules. 
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

In [3]:
# Load the testing CSV file into a DataFrame. 
df = pd.read_csv(config.get_data_path('test_data_.csv'))

#Remove the row where 'BDSPPatientID' is 122501769, as this patient was over 120 years old. 
df = df[df['BDSPPatientID'] != 122501769]

#Save the DataFrame back to a CSV file. 
#df.to_csv(config.get_data_path('test_data_.csv'), index=False)


In [4]:
#Assign the correct variables to test the data. 
matrix = pd.read_csv(config.get_data_path('test_data_.csv'))
matrix = matrix.sample(random_state = 2023, frac = 1, ignore_index=True)  

#Drop the annotation (annot) column to create X_data. 
X_data = matrix.drop('annot', axis=1).reset_index(drop = True)

#Extract the 'BDSPPatientID' and 'annot' columns to create y_data_pre. 
y_data_pre = matrix[['BDSPPatientID', 'annot']].reset_index(drop = True)

assert len(X_data) == len(y_data_pre), "DataFrames must have the same length"
X_data['annot']=y_data_pre['annot']
#print(X_data)

y_data = X_data[['BDSPPatientID', 'annot']]
#print(y_data)
y_data_pre=y_data
#print(y_data_pre)
y_holdout = y_data_pre['annot']
#print(y_holdout)

X_data=X_data.drop(['annot'], axis=1)
X_holdout=X_data.drop(['BDSPPatientID', 'ContactDate', 'hospital', 'Unnamed: 0', 'NoteFileName', 'Site'], axis=1) 
#print(X_holdout)
feature_names = X_holdout.columns.tolist()

In [5]:
#Use random forest to conduct 10 fold nested cross validation. 

#Function to compute confidence intervals using bootstrapping. 
def bootstrap_ci(y_true, y_pred_proba, metric_func, num_bootstrap=1000, alpha=0.05):
    stats = []
    for _ in range(num_bootstrap):
        indices = resample(np.arange(len(y_true)), replace=True)
        y_true_bs = y_true[indices]
        y_pred_proba_bs = y_pred_proba[indices]
        stat = metric_func(y_true_bs, y_pred_proba_bs)
        stats.append(stat)
    lower_bound = np.percentile(stats, 100 * alpha / 2)
    upper_bound = np.percentile(stats, 100 * (1 - alpha / 2))
    return lower_bound, upper_bound

# Function to compute ROC curve for bootstrapping
def bootstrap_roc_curves(y_true, y_pred_proba, num_bootstrap=1000):
    roc_curves = []
    for _ in range(num_bootstrap):
        indices = resample(np.arange(len(y_true)), replace=True)
        y_true_bs = y_true[indices]
        y_pred_proba_bs = y_pred_proba[indices]
        fpr, tpr, _ = roc_curve(y_true_bs, y_pred_proba_bs)
        roc_curves.append((fpr, tpr))
    return roc_curves

# Function to compute Precision-Recall curves for bootstrapping
def bootstrap_pr_curves(y_true, y_pred_proba, num_bootstrap=1000):
    pr_curves = []
    for _ in range(num_bootstrap):
        indices = resample(np.arange(len(y_true)), replace=True)
        y_true_bs = y_true[indices]
        y_pred_proba_bs = y_pred_proba[indices]
        precision, recall, _ = precision_recall_curve(y_true_bs, y_pred_proba_bs)
        pr_curves.append((precision, recall))
    return pr_curves



#Load the models and cutoffs. 
models = []
cutoffs = []
feature_importances_dict = {}

for fold in range(10):
    with open(f'RF_model_train_allhospitals_Notes+ICD+Med_fold{fold+1}.pickle', 'rb') as f:
        res = pickle.load(f, encoding='latin1')
    models.append(res['model'])
    cutoffs.append(res['cutoff'])
    
    #Extract the feature importances. 
    model = res['model']
    if hasattr(model, 'feature_importances_'):
        feature_importances_dict[fold] = model.feature_importances_

#Predict on the holdout set. 
y_pred_proba = np.zeros(X_holdout.shape[0])
y_pred = np.zeros(X_holdout.shape[0])

y_pred_proba = []
for model, cutoff in zip(models, cutoffs):
    yp_proba = model.predict_proba(X_holdout)[:, 1]
    y_pred_proba.append( yp_proba)

/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/sklearn/base.py:348: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
auc_ens = roc_auc_score(y_holdout, np.array(y_pred_proba).mean(axis=0))
print(auc_ens)

0.9791136061085576


In [7]:
aucs_1fold = []
for fold in range(10):
    auc_1fold = roc_auc_score(y_holdout, y_pred_proba[fold])
    aucs_1fold.append(auc_1fold)
    print(auc_1fold)

0.9785685580086906
0.9782244493004797
0.9785999547886368
0.9786225604701981
0.978087559339914
0.9784630648280713
0.9792241227739683
0.9788184763770628
0.977712053851757
0.9786577248637378


In [8]:
from scipy.stats import ttest_1samp

In [9]:
ttest_1samp(aucs_1fold, auc_ens)

TtestResult(statistic=-4.694193588619604, pvalue=0.0011293712458977552, df=9)